# 01. Librerie

In [19]:
%reset -f
import numpy as np
import pandas as pd

# 02. Input

In [20]:
input = r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\99_Modelli\03.output_test"
output = r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\99_Modelli\03.output_test"

df = pd.read_excel(input + r"\02_Univariate_TD_livello.xlsx", sheet_name="Summary_modello")

# 03. Definizione score

In [21]:
# ripristina le etichette replicate da Excel
df["VAR2"] = df["VAR2"].ffill()

#Isolo tasso di infortuni lag
df_2= df[df["VAR2"].eq("L1_tasso_infortuni")].copy()


In [22]:
# Metriche
metrics = ["CORR", "R2_ADJ", "PVAL"]
df_3 = df_2[df_2["METRICA"].isin(metrics)].copy()

# Colonne ATECO
id_cols = ["VAR2", "METRICA"]
ateco_cols = [c for c in df_3.columns if c not in id_cols]
print("ATECO columns:", ateco_cols)

df_3[ateco_cols] = df_3[ateco_cols].apply(pd.to_numeric, errors="coerce")

ATECO columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']


## Calcolo score finale con metodo massimo-minimo e con standardizzazione 

In [23]:
def add_score_row(df_scaled, ateco_cols_in):
    score_vals = (
        df_scaled.set_index("METRICA")
                 .loc[["CORR", "R2_ADJ", "PVAL"], ateco_cols_in]
                 .mean(axis=0)
    )
    score_row = pd.DataFrame(
        [[df_scaled["VAR2"].iloc[0], "SCORE"] + score_vals.tolist()],
        columns=["VAR2", "METRICA"] + ateco_cols_in
    )
    return pd.concat([df_scaled, score_row], ignore_index=True)

# Scaling: MIN-MAX (PVAL invertito)
def row_minmax(s):
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series([pd.NA] * len(s), index=s.index)
    return (s - mn) / (mx - mn)

def scale_minmax(df_in, ateco_cols_in):
    out = df_in.copy()
    for i, row in df_in.iterrows():
        vals = row[ateco_cols_in]
        scaled = row_minmax(vals)
        if row["METRICA"] == "PVAL":
            scaled = 1 - scaled
        out.loc[i, ateco_cols_in] = scaled
    return out

# Esecuzione
df_mm_A = scale_minmax(df_3, ateco_cols)
df_mm_A = add_score_row(df_mm_A, ateco_cols)

## Export

In [27]:
# ---------- EXPORT ----------
out_file = output + r"\03_Definizione_score.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:

    # Dataframe 
    df_mm_A.to_excel(writer, sheet_name="Score_MinMax_A", index=False)